# Reliability measures

## Join R stats, preprocessing

In [4]:
import pandas as pd
from llm_audit import BASE_DIR
from llm_audit.datasets.util import get_dataset_label_class_map

# Columns to extract
fit_indices = [
    "chi.sq.chisq",
    "df.df",
    "chi.sq.df.ratio.chisq",
    "RMSEA.rmsea",
    "RMSEA_CI_lower.rmsea.ci.lower",
    "RMSEA_CI_upper.rmsea.ci.upper",
    "SRMR.srmr",
    "CFI.cfi",
    "GFI.gfi",
]

datasets_to_exclude = ["BFI10", "PISD", "SDO7", "CW", "PI"]
dataset_labels = [
    d for d in list(get_dataset_label_class_map(filter_test_and_vignettes=True).keys()) if d not in datasets_to_exclude
]
results = {"open": [], "closed": []}

for approach in ["open", "closed"]:
    for dataset_label in dataset_labels:
        file_path = BASE_DIR / "eval" / "data" / "reliability" / "R2" / f"{dataset_label}_{approach}_stats.csv"

        try:
            df = pd.read_csv(file_path)
            df = df[fit_indices]
            df.insert(0, "dataset", dataset_label)
            results[approach].append(df)

        except FileNotFoundError:
            print(f"Warning: File not found - {file_path}")
            # Create empty row with dataset name
            empty_row = pd.DataFrame(
                [[dataset_label] + [None] * len(fit_indices)],
                columns=["dataset"] + fit_indices,
            )
            results[approach].append(empty_row)
        except Exception as e:
            print(f"Error processing {dataset_label}_{approach}: {e}")
            # Create empty row with dataset name
            empty_row = pd.DataFrame(
                [[dataset_label] + [None] * len(fit_indices)],
                columns=["dataset"] + fit_indices,
            )
            results[approach].append(empty_row)

# Combine and save
for approach in ["open", "closed"]:
    if results[approach]:
        combined_df = pd.concat(results[approach], ignore_index=True)
        output_path = BASE_DIR / "eval" / "data" / "reliability" / f"cfa_{approach}_stats.csv"
        combined_df.to_csv(output_path, index=False)

Error processing VSA_open: "None of [Index(['chi.sq.chisq', 'df.df', 'chi.sq.df.ratio.chisq', 'RMSEA.rmsea',\n       'RMSEA_CI_lower.rmsea.ci.lower', 'RMSEA_CI_upper.rmsea.ci.upper',\n       'SRMR.srmr', 'CFI.cfi', 'GFI.gfi'],\n      dtype='object')] are in the [columns]"


/tmp/ipykernel_161272/2656870694.py:54: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  combined_df = pd.concat(results[approach], ignore_index=True)


### Generate CFA tables

In [5]:
import pandas as pd
from llm_audit import BASE_DIR


rel_cfa_open_path = BASE_DIR / "eval" / "data" / "reliability" / "cfa_open_stats.csv"
rel_cfa_closed_path = BASE_DIR / "eval" / "data" / "reliability" / "cfa_closed_stats.csv"
rel_cfa_open = pd.read_csv(rel_cfa_open_path)
rel_cfa_closed = pd.read_csv(rel_cfa_closed_path)

# Column name mapping for better display
column_names = {
    "dataset": "Dataset",
    "chi.sq.chisq": "$\\chi^2$",
    "df.df": "df",
    "chi.sq.df.ratio.chisq": "$\\chi^2$/df",
    "RMSEA_combined": "RMSEA (CI)",
    "SRMR.srmr": "SRMR",
    "CFI.cfi": "CFI",
    "GFI.gfi": "GFI",
}


def format_value(val, col_name=""):
    """Format numeric values for display"""
    if pd.isna(val):
        return "--"
    try:
        val_float = float(val)
        # Chi-square and df as integers
        if col_name in ["$\\chi^2$", "df"] or val_float > 100:
            return f"{int(val_float)}"
        # Other metrics with 3 decimal places
        else:
            return f"{val_float:.3f}"
    except:
        return str(val)


def create_latex_table(df, caption, label):
    # Create RMSEA combined column
    df = df.copy()

    def combine_rmsea(row):
        rmsea = row["RMSEA.rmsea"]
        lower = row["RMSEA_CI_lower.rmsea.ci.lower"]
        upper = row["RMSEA_CI_upper.rmsea.ci.upper"]

        if pd.isna(rmsea):
            return "--"

        rmsea_str = f"{float(rmsea):.3f}"

        if pd.isna(lower) or pd.isna(upper):
            return rmsea_str

        lower_str = f"{float(lower):.3f}"
        upper_str = f"{float(upper):.3f}"

        return f"{rmsea_str} [{lower_str}, {upper_str}]"

    df["RMSEA_combined"] = df.apply(combine_rmsea, axis=1)

    # Select and reorder columns
    columns_to_keep = [
        "dataset",
        "chi.sq.chisq",
        "df.df",
        "chi.sq.df.ratio.chisq",
        "RMSEA_combined",
        "SRMR.srmr",
        "CFI.cfi",
        "GFI.gfi",
    ]
    df_display = df[columns_to_keep]

    # Rename columns
    df_display = df_display.rename(columns=column_names)

    # Format numeric columns (skip RMSEA_combined as it's already formatted)
    for col in df_display.columns:
        if col not in ["Dataset", "RMSEA (CI)"]:
            df_display[col] = df_display[col].apply(lambda x: format_value(x, col))

    # Create LaTeX table
    latex = []
    latex.append("\\begin{table}[t]")
    latex.append("\\centering")
    latex.append("\\small")
    latex.append(f"\\caption{{{caption}}}")
    latex.append(f"\\label{{{label}}}")

    # Determine column alignment
    n_cols = len(df_display.columns)
    col_align = "l" + "r" * (n_cols - 1)

    latex.append(f"\\begin{{tabular}}{{{col_align}}}")
    latex.append("\\toprule")

    # Header
    headers = " & ".join(df_display.columns) + " \\\\"
    latex.append(headers)
    latex.append("\\midrule")

    # Data rows
    for idx, (_, row) in enumerate(df_display.iterrows()):
        row_str = "\\rowcolor{Gray}" if idx % 2 == 0 else ""
        row_str += " & ".join(str(val) for val in row.values) + " \\\\"
        latex.append(row_str)

    latex.append("\\bottomrule")
    latex.append("\\end{tabular}")
    latex.append("\\end{table}")

    return "\n".join(latex)


# Generate tables
latex_open = create_latex_table(rel_cfa_open, "CFA Model Fit Indices for Open Approach", "tab:cfa-open")

latex_closed = create_latex_table(rel_cfa_closed, "CFA Model Fit Indices for Closed Approach", "tab:cfa-closed")

# Save
output_dir = BASE_DIR / "eval" / "data" / "reliability"
with open(output_dir / "cfa_open_latex.tex", "w") as f:
    f.write(latex_open)

with open(output_dir / "cfa_closed_latex.tex", "w") as f:
    f.write(latex_closed)


print("LaTeX tables generated!")
print(f"\nFiles saved to: {output_dir}")

LaTeX tables generated!

Files saved to: /root/llm-audit/eval/data/reliability


### Generate reliability tables

In [6]:
""" Generate LaTeX tables for reliability statistics from R1 and R2 data.

This script processes reliability data from two sources:
- R1: Contains alpha, omega_h, omega_tot statistics with confidence intervals
- R2: Contains Composite Reliability (CR) statistics from CFA

It generates two LaTeX tables (open and closed approach) with the following columns:
- Dataset name (multirow for datasets with multiple factors)
- Factor name (with indentation)
- Cronbach's alpha with 95% CI
- Omega hierarchical
- Omega total
- Composite Reliability (CR)
"""

import pandas as pd
from pathlib import Path
from llm_audit import BASE_DIR
from llm_audit.datasets.util import get_dataset_label_class_map, get_dataset_by_label

# Columns to extract from R1
static_r1_columns = ["factor", "alpha", "a.l.ci95", "a.u.ci95", "omega_h", "omega.tot"]

datasets_to_exclude = ["BFI10", "PISD", "SDO7", "CW", "PI"]
dataset_labels = [
    d for d in list(get_dataset_label_class_map(filter_test_and_vignettes=True).keys()) if d not in datasets_to_exclude
]

print(f"Processing {len(dataset_labels)} datasets: {', '.join(dataset_labels)}")
print("=" * 80)


def extract_r1_data(file_path, factors, dataset_label) -> dict | None:
    """Extract R1 statistics from CSV file.

    Args:
        file_path: Path to R1 statistics CSV file
        factors: List of factor names for multi-factor datasets (empty for single-factor)
        dataset_label: Label of the dataset

    Returns:
        Dictionary mapping factor names to their statistics, or None if file doesn't exist
    """
    if not file_path.exists():
        print(f"R1 file not found: {file_path.name}")
        return None

    try:
        df = pd.read_csv(file_path)
    except Exception as e:
        print(f"Error reading R1 file {file_path.name}: {e}")
        return None

    # Prepare results dictionary
    results = {}

    if not factors:
        # Single factor dataset
        row = df[df["factor"] == dataset_label]
        if not row.empty:
            results[dataset_label] = row.iloc[0].to_dict()
    else:
        # Multi-factor dataset
        for factor in factors:
            row = df[df["factor"] == factor]
            if not row.empty:
                results[factor] = row.iloc[0].to_dict()

        # Add joint factor if exists
        joint_row = df[df["factor"] == "joint.factor"]
        if not joint_row.empty:
            results["joint.factor"] = joint_row.iloc[0].to_dict()
    return results


def extract_r2_data(file_path, factors, dataset_label) -> dict | None:
    """Extract CR (Composite Reliability) from R2 CSV file.

    Args:
        file_path: Path to R2 statistics CSV file
        factors: List of factor names for multi-factor datasets (empty for single-factor)
        dataset_label: Label of the dataset

    Returns:
        Dictionary mapping factor names to their CR values, or None if file doesn't exist
    """
    if not file_path.exists():
        print(f"R2 file not found: {file_path.name}")
        return None

    try:
        df = pd.read_csv(file_path)
    except Exception as e:
        print(f"Error reading R2 file {file_path.name}: {e}")
        return None

    if df.empty:
        return None

    results = {}
    row = df.iloc[0]

    if not factors:
        # Single factor dataset
        cr_col = f"CR_{dataset_label}"
        if cr_col in row.index:
            results[dataset_label] = row[cr_col]
    else:
        # Multi-factor dataset
        for factor in factors:
            cr_col = f"CR_{factor}"
            if cr_col in row.index:
                results[factor] = row[cr_col]

        # Add joint factor (maps to CR_total)
        if "CR_total" in row.index:
            results["joint.factor"] = row["CR_total"]
    return results


def format_value(value, decimals=3) -> str:
    """Format numeric value for LaTeX table."""
    if pd.isna(value) or value == "NA":
        return "---"
    try:
        return f"{float(value):.{decimals}f}"
    except (ValueError, TypeError):
        return "---"


def generate_latex_table(approach, dataset_labels) -> str:
    """Generate LaTeX table for given approach (open or closed)."""

    table_rows = []
    datasets_processed = 0

    for dataset_label in dataset_labels:
        # print(f"\nProcessing dataset: {dataset_label}")

        try:
            dataset = get_dataset_by_label(dataset_label=dataset_label)
            factors = dataset.get_factors()
        except Exception as e:
            print(f"Error loading dataset {dataset_label}: {e}")
            continue

        # print(f" Factors: {factors if factors else '[single-factor]'}")

        file_path_r1 = BASE_DIR / "eval" / "data" / "reliability" / "R1" / f"{dataset_label}_{approach}_stats.csv"
        file_path_r2 = BASE_DIR / "eval" / "data" / "reliability" / "R2" / f"{dataset_label}_{approach}_stats.csv"

        r1_data = extract_r1_data(file_path_r1, factors, dataset_label)
        r2_data = extract_r2_data(file_path_r2, factors, dataset_label)

        # If both files missing, skip dataset
        if r1_data is None and r2_data is None:
            print(f"Skipping {dataset_label} - no data files found")
            continue

        datasets_processed += 1

        # Determine which factors to display
        if not factors:
            factor_list = [dataset_label]
        else:
            factor_list = factors + ["joint.factor"]

        for idx, factor in enumerate(factor_list):
            # Extract values from R1
            if r1_data and factor in r1_data:
                r1_row = r1_data[factor]
                alpha = format_value(r1_row.get("alpha"))
                alpha_lower = format_value(r1_row.get("a.l.ci95"))
                alpha_upper = format_value(r1_row.get("a.u.ci95"))
                omega_h = format_value(r1_row.get("omega_h"))
                omega_tot = format_value(r1_row.get("omega.tot"))
            else:
                alpha = alpha_lower = alpha_upper = omega_h = omega_tot = "---"

            # Extract CR from R2
            if r2_data and factor in r2_data:
                cr = format_value(r2_data[factor])
            else:
                cr = "---"

            # Format dataset and factor names
            if idx == 0:
                # First row: show dataset label with multirow
                if len(factor_list) > 1:
                    dataset_col = f"\\multirow{{{len(factor_list)}}}{{*}}{{{dataset_label}}}"
                else:
                    dataset_col = f"{dataset_label}"
            else:
                # Subsequent rows: empty dataset column
                dataset_col = ""

            # Format factor name (indent all factors)
            factor_name = f"\\quad {factor}"

            # Combine alpha and CI into single column: "alpha [lower, upper]"
            alpha_with_ci = f"{alpha} [{alpha_lower}, {alpha_upper}]"

            # Create table row
            row = f"{dataset_col} & {factor_name} & {alpha_with_ci} & {omega_h} & {omega_tot} & {cr} \\\\"
            table_rows.append(row)

        table_rows.append("\\hline")

    print(f"\n✓ Successfully processed {datasets_processed} datasets for {approach} approach")

    # Construct full LaTeX table
    latex_table = (
        """\\begin{table}[htbp]
\\centering
\\caption{Reliability Statistics for """
        + approach.capitalize()
        + """ Approach}
\\label{tab:reliability-"""
        + approach
        + """}
\\begin{tabular}{llrrrr}
\\hline
Dataset & Factor & $\\alpha$ (CI) & $\\omega_h$ & $\\omega_{tot}$ & CR \\\\
\\hline
"""
    )

    latex_table += "\n".join(table_rows)
    latex_table += """\\end{tabular}
\\end{table}
"""
    return latex_table


# Generate tables for both approaches
results = {"open": [], "closed": []}

for approach in ["open", "closed"]:
    latex_table = generate_latex_table(approach, dataset_labels)
    results[approach] = latex_table

    # Save to file
    output_file = BASE_DIR / "eval" / "data" / "reliability" / f"reliability_table_{approach}.tex"
    output_file.parent.mkdir(parents=True, exist_ok=True)

    with open(output_file, "w") as f:
        f.write(latex_table)

    print(f"Generated LaTeX table for {approach} approach: {output_file}")
    print("=" * 80)

Processing 15 datasets: F, LAS, D, A, AA, RWA, RWA3D, KSA3, ACT, VSA, ASC, APC, CSM, DW, BDW

✓ Successfully processed 15 datasets for open approach
Generated LaTeX table for open approach: /root/llm-audit/eval/data/reliability/reliability_table_open.tex

✓ Successfully processed 15 datasets for closed approach
Generated LaTeX table for closed approach: /root/llm-audit/eval/data/reliability/reliability_table_closed.tex
